In [1]:
import json
import statistics
import os
from glob import glob
from statsmodels.stats.power import TTestIndPower
import numpy as np
import scipy.stats as stats
from scipy.stats import shapiro
import numpy as np
import math

In [2]:
def calculate_sample_size(delta, std_dev, alpha=0.05, power=0.8):
    effect_size = delta / std_dev
    analysis = TTestIndPower()
    n = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, alternative='two-sided')
    return int(np.ceil(n))

def get_sample_size(margin, std_dev):
    required_n = calculate_sample_size(margin, std_dev)
    print(f"Required sample size per group (two-sided test): {required_n}")

def check_normality(data):
    stat, p = shapiro(data)
    if p < 0.05:
        return False
    else:
        return True
        
def read_files(files_path, num_keys):
    txn_files = glob(files_path + f"/**/ycsbt_uni_{num_keys}_txn.json", recursive=True)
    hyb_files = glob(files_path + f"/**/ycsbt_uni_{num_keys}_hyb.json", recursive=True)

    res_dict = {}
    txn_res = []
    for file in hyb_files:
        with open(file, 'r') as f:
            txn_res.append(json.load(f))
    hyb_res = []
    for file in txn_files:
        with open(file, 'r') as f:
            hyb_res.append(json.load(f))
    
    txn_latency_mean = []
    txn_latency_95 = []
    txn_latency_99 = []
    txn_throughput_mean = []
    txn_consistency = []
    txn_missed = []
    for res in txn_res:
        txn_latency_mean.append(res["latency (ms)"]["mean"])
        txn_latency_95.append(res["latency (ms)"]["95"])
        txn_latency_99.append(res["latency (ms)"]["99"])
        txn_throughput_mean.append(res["throughput"]["avg"])
        txn_consistency.append(res["are_we_consistent"])
        txn_missed.append(res["missed messages"])
    
    hyb_latency_mean = []
    hyb_latency_95 = []
    hyb_latency_99 = []
    hyb_throughput_mean = []
    hyb_consistency = []
    hyb_missed = []
    for res in hyb_res:
        hyb_latency_mean.append(res["latency (ms)"]["mean"])
        hyb_latency_95.append(res["latency (ms)"]["95"])
        hyb_latency_99.append(res["latency (ms)"]["99"])
        hyb_throughput_mean.append(res["throughput"]["avg"])
        hyb_consistency.append(res["are_we_consistent"])
        hyb_missed.append(res["missed messages"])
        
    res_dict = {
        "Latency mean txn": txn_latency_mean,
        "Latency mean hyb": hyb_latency_mean,
        "Latency 95p txn": txn_latency_95,
        "Latency 95p hyb": hyb_latency_95,
        "Latency 99p txn": txn_latency_99,
        "Latency 99p hyb": hyb_latency_99,
        "Throughput txn": txn_throughput_mean,
        "Throughput hyb": hyb_throughput_mean,
        "Consistency txn": txn_consistency,
        "Consistency hyb": hyb_consistency ,
        "Missed txn": txn_missed,
        "Missed hyb": hyb_missed
    }
    return res_dict

def get_std_dev(values):
    return statistics.stdev(values)

def non_inferiority_test_parametric(data_A, data_B, delta, alpha = 0.05):
    n_A = len(data_A)
    n_B = len(data_B)

    mean_A = np.mean(data_A)
    print(f"System 1 mean: {mean_A}")
    mean_B = np.mean(data_B)
    print(f"System 2 mean: {mean_B}")
    mean_diff = mean_B - mean_A

    var_A = np.var(data_A, ddof=1)
    var_B = np.var(data_B, ddof=1)

    se_diff = np.sqrt(var_A / n_A + var_B / n_B)

    df_num = (var_A / n_A + var_B / n_B) ** 2
    df_den = ((var_A / n_A) ** 2) / (n_A - 1) + ((var_B / n_B) ** 2) / (n_B - 1)
    df = df_num / df_den

    t_stat = (mean_diff + delta) / se_diff
    p_value = 1 - stats.t.cdf(t_stat, df)

    print(f"p-value = {p_value}")
    if p_value < alpha or math.isnan(p_value):
        return True
    else:
        return False

def non_inferiority_test_nonparametric(data1, data2, delta, n_bootstrap=10000, alpha=0.05):
    observed_diff = np.mean(data2) - np.mean(data1)
    print(f"System 1 mean: {np.mean(data1):.3f}")
    print(f"System 2 mean: {np.mean(data2):.3f}")
    print(f"Observed mean difference (B - A): {observed_diff:.3f}")
    boot_diffs = []
    for _ in range(n_bootstrap):
        sample_A = np.random.choice(data1, size=len(data1), replace=True)
        sample_B = np.random.choice(data2, size=len(data2), replace=True)
        boot_diffs.append(np.mean(sample_B) - np.mean(sample_A))
    boot_diffs = np.array(boot_diffs)
    p_value = np.mean(boot_diffs <= -delta)
    print(f"p-value = {p_value}")
    if p_value < alpha:
        return True
    else:
        return False

def get_non_inferiority(results, margin, measure):
    pure_txn = results[f"{measure} txn"]
    hybrid = results[f"{measure} hyb"]
    print(f"Pure txn = {pure_txn}, hybrid = {hybrid}")
    is_normal = check_normality(pure_txn) and check_normality(hybrid)
    if is_normal:
        res = non_inferiority_test_parametric(pure_txn, hybrid, margin)
    else:
        print("Non-parametric test needed")
        res = non_inferiority_test_nonparametric(pure_txn, hybrid, margin)
    if res:
        print(f"System B is non-inferior wrt {measure}")
    else:
        print(f"System B is inferior wrt {measure}")
        
def get_consistency_and_missed(results):
    print(f"Styx Consistency: {results['Consistency txn']}")
    print(f"H-Styx Consistency: {results['Consistency hyb']}")
    print(f"Styx Missed Messages: {results['Missed txn']}")
    print(f"H-Styx Missed Messages: {results['Missed hyb']}")

In [8]:
results = read_files("exp1-keys", 10000)
print("LATENCY")
get_non_inferiority(results, 0.1, "Latency mean")
get_non_inferiority(results, 0.1, "Latency 95p")
get_non_inferiority(results, 1, "Latency 99p")
print("THROUGHPUT")
get_non_inferiority(results, 1, "Throughput")
get_consistency_and_missed(results)

LATENCY
Pure txn = [4.512243309585105, 4.73986641411624, 4.684794761220252, 4.550742277915236, 4.718511279095279, 4.654442004392094, 4.613706667731056, 4.613332002794133, 4.672406568629407, 4.5393244969658255], hybrid = [4.656408414161977, 4.650994250119789, 4.735438007947443, 4.587380481865182, 4.495599065924196, 4.636363636363637, 4.577621651830266, 4.703292755936939, 4.643660031545111, 4.624812825683311]
System 1 mean: 4.629936978244463
System 2 mean: 4.631157112137786
p-value = 0.0030115682981946
System B is non-inferior wrt Latency mean
Pure txn = [7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0], hybrid = [7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0]
System 1 mean: 7.0
System 2 mean: 7.0
p-value = nan
System B is non-inferior wrt Latency 95p
Pure txn = [9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0], hybrid = [9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0]
System 1 mean: 9.0
System 2 mean: 9.0
p-value = nan
System B is non-inferior wrt Latency 99p
THROUGHPUT
Pure txn 

C:\Users\smrut\AppData\Local\Temp\ipykernel_19072\934890619.py:96: RuntimeWarning: invalid value encountered in scalar divide
  df = df_num / df_den
C:\Users\smrut\AppData\Local\Temp\ipykernel_19072\934890619.py:98: RuntimeWarning: divide by zero encountered in scalar divide
  t_stat = (mean_diff + delta) / se_diff


In [9]:
results = read_files("exp1-keys", 100000)
print("LATENCY")
get_non_inferiority(results, 0.1, "Latency mean")
get_non_inferiority(results, 0.1, "Latency 95p")
get_non_inferiority(results, 1, "Latency 99p")
print("THROUGHPUT")
get_non_inferiority(results, 5, "Throughput")
get_consistency_and_missed(results)

LATENCY
Pure txn = [4.517324013979032, 4.675799816271917, 4.674230922892529, 4.6675322081294315, 4.66175531065325, 4.652756424335603, 4.701253092824647, 4.575646272083042, 4.780793736393233, 4.674638664856664], hybrid = [4.618715808808849, 4.693174102328554, 4.6478484957449355, 4.541390596311966, 4.6651833818892925, 4.6970452686391475, 4.666327162342979, 4.703949339779061, 4.7128511249975045, 4.673873513837546]
System 1 mean: 4.658173046241935
System 2 mean: 4.662035879467984
p-value = 0.0007845055769222231
System B is non-inferior wrt Latency mean
Pure txn = [7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0], hybrid = [7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0]
System 1 mean: 7.0
System 2 mean: 7.0
p-value = nan
System B is non-inferior wrt Latency 95p
Pure txn = [9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 10.0, 9.0], hybrid = [9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 10.0, 10.0, 9.0]
Non-parametric test needed
System 1 mean: 9.100
System 2 mean: 9.200
Observed mean difference (B - A): 

C:\Users\smrut\AppData\Local\Temp\ipykernel_19072\934890619.py:96: RuntimeWarning: invalid value encountered in scalar divide
  df = df_num / df_den
C:\Users\smrut\AppData\Local\Temp\ipykernel_19072\934890619.py:98: RuntimeWarning: divide by zero encountered in scalar divide
  t_stat = (mean_diff + delta) / se_diff


p-value = 0.0
System B is non-inferior wrt Latency 99p
THROUGHPUT
Pure txn = [981.8627450980392, 981.843137254902, 981.5686274509804, 981.6666666666666, 982.1176470588235, 982.0196078431372, 982.6666666666666, 982.2549019607843, 981.7058823529412, 982.1960784313726], hybrid = [982.0784313725491, 981.843137254902, 981.5294117647059, 982.5098039215686, 982.6274509803922, 982.8039215686274, 981.8235294117648, 981.5490196078431, 982.1372549019608, 981.2745098039215]
System 1 mean: 981.9901960784313
System 2 mean: 982.0176470588236
p-value = 1.5432100042289676e-14
System B is non-inferior wrt Throughput
Styx Consistency: [True, True, True, True, True, True, True, True, True, True]
H-Styx Consistency: [True, True, True, True, True, True, True, True, True, True]
Styx Missed Messages: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
H-Styx Missed Messages: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [6]:
results = read_files("exp1-sstime", 10000)
print("LATENCY")
get_non_inferiority(results, 0.1, "Latency mean")
get_non_inferiority(results, 0.1, "Latency 95p")
get_non_inferiority(results, 1, "Latency 99p")
print("THROUGHPUT")
get_non_inferiority(results, 5, "Throughput")
get_consistency_and_missed(results)

LATENCY
Pure txn = [4.569845674699036, 4.646298144083017, 4.673877901291236, 4.564048844726446, 4.605217981475567, 4.5493053337591824, 4.747464867454488, 4.630811285366535, 4.621590863714411, 4.632799712563376], hybrid = [4.630492038788459, 4.658035429300989, 4.649468074489532, 4.574903228381021, 4.620578136229338, 4.6406833103833645, 4.699086266060171, 4.630641780616828, 4.603606302042772, 4.680838323353293]
System 1 mean: 4.624126060913329
System 2 mean: 4.638833288964577
p-value = 4.6712119339376024e-05
System B is non-inferior wrt Latency mean
Pure txn = [7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0], hybrid = [7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0]
System 1 mean: 7.0
System 2 mean: 7.0
p-value = nan
System B is non-inferior wrt Latency 95p
Pure txn = [9.0, 9.0, 10.0, 9.0, 9.0, 9.0, 10.0, 9.0, 9.0, 9.0], hybrid = [9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0]
Non-parametric test needed
System 1 mean: 9.200
System 2 mean: 9.000
Observed mean difference (B - A): -0

C:\Users\smrut\AppData\Local\Temp\ipykernel_19072\934890619.py:96: RuntimeWarning: invalid value encountered in scalar divide
  df = df_num / df_den
C:\Users\smrut\AppData\Local\Temp\ipykernel_19072\934890619.py:98: RuntimeWarning: divide by zero encountered in scalar divide
  t_stat = (mean_diff + delta) / se_diff


p-value = 0.0
System B is non-inferior wrt Latency 99p
THROUGHPUT
Pure txn = [982.1372549019608, 982.5490196078431, 982.4901960784314, 982.7058823529412, 982.2745098039215, 982.2745098039215, 982.2745098039215, 982.7058823529412, 982.0784313725491, 982.3137254901961], hybrid = [982.7058823529412, 982.9019607843137, 982.3725490196078, 982.7058823529412, 982.1960784313726, 982.5294117647059, 982.8235294117648, 982.2549019607843, 981.9411764705883, 982.3529411764706]
System 1 mean: 982.3803921568627
System 2 mean: 982.4784313725492
p-value = 0.0
System B is non-inferior wrt Throughput
Styx Consistency: [True, True, True, True, True, True, True, True, True, True]
H-Styx Consistency: [True, True, True, True, True, True, True, True, True, True]
Styx Missed Messages: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
H-Styx Missed Messages: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
